# 00 — Setup Unity Catalog

Crea el catalogo y los tres schemas: `raw`, `trusted`, `refined`.

In [ ]:
%run ../config/pipeline_config

In [ ]:
import logging
import time

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")
logger = logging.getLogger("setup_unity_catalog")

start_time = time.time()

## Crear Catalogo

In [ ]:
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    spark.sql(f"COMMENT ON CATALOG {CATALOG_NAME} IS 'Pipeline Medallion para datos de taxis amarillos de NYC (enero 2023). Propietario: Leonardo Aguilera.'")
    logger.info(f"Catalogo '{CATALOG_NAME}' creado o ya existente.")
except Exception as e:
    logger.error(f"Error creando catalogo: {e}")
    raise

## Crear Schemas (Raw, Trusted, Refined)

In [ ]:
schemas = {
    RAW_SCHEMA: "Capa de datos crudos sin transformacion. Preserva el formato original del dataset.",
    TRUSTED_SCHEMA: "Capa de datos validados, limpios y enriquecidos con zonas de taxi.",
    REFINED_SCHEMA: "Capa de KPIs de negocio, reportes de calidad y metricas de ejecucion.",
}

for schema_name, description in schemas.items():
    try:
        fqn = f"{CATALOG_NAME}.{schema_name}"
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {fqn}")
        spark.sql(f"COMMENT ON SCHEMA {fqn} IS '{description}'")
        logger.info(f"Schema '{fqn}' creado correctamente.")
    except Exception as e:
        logger.error(f"Error creando schema '{schema_name}': {e}")
        raise

# Crear Volume para almacenamiento temporal (compatible con shared clusters UC)
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {TMP_VOLUME}")
    logger.info(f"Volume '{TMP_VOLUME}' creado o ya existente.")
except Exception as e:
    logger.error(f"Error creando volume: {e}")
    raise

## Verificacion

In [ ]:
df_schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG_NAME}")
df_schemas.show()

elapsed = round(time.time() - start_time, 2)
logger.info(f"Setup completado en {elapsed}s. Catalogo: {CATALOG_NAME}, Schemas: {list(schemas.keys())}")